# MMM Colab Runner

Clones the repo, installs deps, mounts credentials, and runs `scripts/run_model.py`.

**Before running:** push any local changes to GitHub — this notebook always pulls fresh from main.

**To run a different client or mode:** edit `CLIENT` and `MODE` in Cell 5, then Runtime → Run all.

In [ ]:
# Cell 1: Clone repo (safe to re-run — pulls latest if already cloned)
import os
REPO = "/content/Meridian-MMM-System"
if os.path.isdir(REPO):
    !git -C {REPO} pull origin main
else:
    !git clone https://github.com/Russ-Moonride/Meridian-MMM-System.git {REPO}
%cd {REPO}

In [ ]:
# Cell 2: Install dependencies
!pip install -r requirements.txt -q

In [ ]:
# Cell 3: Download service account credentials via gdown
import gdown
from google.colab import drive
drive.mount('/content/drive')

file_id = '1Nnvt5h7QW6hC1R-sCiqXSgKvLclR767z'
destination_path = '/content/Meridian-MMM-System/service_account.json'

gdown.download(id=file_id, output=destination_path, quiet=False)

In [ ]:
# Cell 4: Set environment variables
import os
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = \
    '/content/Meridian-MMM-System/service_account.json'
os.environ['CUDA_VISIBLE_DEVICES'] = ''

In [ ]:
# Cell 5: Run the model — edit CLIENT and MODE as needed
CLIENT = "northspore"
MODE = "prod"

!python scripts/run_model.py --client {CLIENT} --mode {MODE}

In [ ]:
# Cell 6 (optional): Re-run BQ write only — skips MCMC, uses outputs already on disk
# Use this to test or retry a failed BQ write without re-running the model.
import sys, importlib
sys.path.insert(0, '/content/Meridian-MMM-System')

import src.bq_writer
importlib.reload(src.bq_writer)
from src.bq_writer import write_run, PROJECT, DATASET
from google.cloud import bigquery

# Drop all mmm_results tables so they're recreated with the correct schema
bq = bigquery.Client(project=PROJECT)
for table in ["runs", "diagnostics", "contributions"]:
    bq.delete_table(f"{PROJECT}.{DATASET}.{table}", not_found_ok=True)
    print(f"  Dropped {DATASET}.{table}")

RUN_ID = f"{MODE}_{__import__('datetime').date.today()}"

write_run(
    client_id=CLIENT,
    run_id=RUN_ID,
    outputs_dir=f"outputs/{CLIENT}",
)

In [ ]:
# Cell 6: Confirm outputs and BigQuery write
!ls outputs/{CLIENT}/
print("Run complete. Check BigQuery mmm_results dataset.")